# Vectorized String Operations in Pandas (Session 22)

Programming languages are great at numbers and terrible at text — at least, they are slow at text unless you use the right toolkit. Consider a simple request like *uppercase every passenger name on the Titanic*. With plain Python you would loop over each string: `return [x.upper() for x in names]`. That works for a hundred names, but real datasets hold millions of rows, and Python's per-element loop plus its fragile handling of missing values makes the approach slow and crash-prone — try `None.upper()` and you get a `TypeError`. This session shows how Pandas turns text wrangling into the same kind of fast, safe, whole-column operation that NumPy does for numbers: *vectorized string operations*.

The central character is the **`.str` accessor**. Attach it to a Series of strings (`s.str.upper()`, `s.str.startswith('c')`) and every element is processed in one compiled, vectorised pass — and missing values stop being landmines. `None` and `NaN` are simply skipped and stay missing, rather than crashing your pipeline. The session practices directly on the Titanic `Name` column, which conveniently follows the famous pattern `Lastname, Title. Firstname`: you will split on commas, pull out the surname, strip whitespace, re-split the given name to capture a `title` (Mr., Mrs., Miss., Dr., ...) and a `firstname`, standardise variant titles with `replace`, filter rows with `startswith`/`endswith`/`isdigit`, and finally dip into **regular expressions** — the pattern language that turns `.contains('john', case=False)` into an instant case-insensitive search and lets `^[^aeiouAEIOU].+[^aeiouAEIOU]$` find surnames that start and end with consonants. Even string slicing (`df['Name'].str[::-1]`) works element-wise. By the end you will clean messy text in one expressive line instead of a slow loop.

## 0. Setup

Import the usual suspects to begin.

In [16]:
import pandas as pd
import numpy as np

# 1. Vectorized Operations

Vectorization is the reason Pandas can chew through hundreds of thousands of strings without breaking a sweat: one operation applied to the whole column at once, in compiled code, instead of a Python `for` loop touching one element at a time.

# ⚡ Vectorized String Operations in Pandas

**Topic:** 07.07 · **Level:** 🟠 Intermediate · **Type:** THEORY + CODE

## 📖 Vectorized operations

```python
a = np.array([1, 2, 3, 4])
a * 4                    # elementwise → [4,8,12,16]
```

- Vectorized = one operation on every element, no loop
- Numpy/pandas core speed
- Direct Python string operations fail on None/NaN


## 🔬 Deep Dive : Vectorized ops — pandas solves python's string slowness

```python
a = np.array([1,2,3,4]); a * 4        # fast elementwise — vectorized

s = ['cat','mat',None,'rat','sitting','batting']     # plain python list
# [x.upper() for x in s]  -> None.upper() crashes TypeError!
```

- Vectorized: op applied to WHOLE array at once (compiled C)
- Python loop: per-element, slow + error-prone
- **Pandas `.str`**: missing values (None/NaN) auto-ignored → no crash
```python
s2 = pd.Series(['cat',None,'rat'])
s2.str.upper()    # NaN stays NaN, rest uppercase
```
- .str = vectorized string methods on Series
- Chaining: s.str.lower().str.strip()...
- NaN propagation handled for you
- This is why pandas over raw python for text columns

## 1.1 What 'vectorized' really means

The NumPy example makes the idea concrete. `a = np.array([1,2,3,4])` then `a * 4` returns `array([4, 8, 12, 16])` — every element multiplied in a single expression, with the loop hidden inside C-level machine code. Compare that with a Python list comprehension; both produce the same numbers, but the NumPy version runs orders of magnitude faster on large arrays and the code is shorter and more readable. This is the same philosophy Pandas inherits for tabular and textual work.

The vulnerability appears the moment text holds anything unexpected. The cell constructs `s = ['cat','mat',None,'rat']` and tries `[i.startswith('c') for i in s]`. The list comprehension walks element-by-element; when it reaches `None`, calling `.startswith` on it raises a `TypeError` — Python objects are *objects* and `None` has no string methods. In the real world missing values are everywhere: blank cells, `None`, `NaN`, and empty strings all arrive from CSVs. A technique that explodes on the first missing value is unusable at scale. The cure arrives in the next section: Pandas' `.str` accessor performs the same logical operation but is deliberately built to tolerate missing data, so a column with a few holes still processes safely in one go.

In [18]:
# What are vectorized operations
a = np.array([1,2,3,4])
a * 4

array([ 4,  8, 12, 16])

In [19]:
# problem in vectorized opertions in vanilla python
s = ['cat','mat',None,'rat']

[i.startswith('c') for i in s]

AttributeError: ignored

# 2. The .str Accessor

The `.str` accessor is Pandas' doorway to every vectorised string method. Write `series.str.method(...)` and the method runs across the whole column at once.

### AI/ML relevance
Shapes, dtypes, vectorization, indexing, and clean tabular ops are critical for reliable data pipelines.

## 📖 .str accessor

```python
s = pd.Series(['cat', 'mat', None, 'rat'])
s.str.upper()    # nan/None handled automatically
```

- `.str` — accesses string methods vectorized
- Missing values (None/NaN) are skipped automatically
- Much faster and safer than Python loops
- Every string method works with `.str`


## 2.1 String methods that never crash on missing data

`pd.Series(['cat','mat',None,'rat'])` followed by `.str.startswith('c')` returns a boolean Series: `True` for `cat`, `False` for `mat` and `rat`, and for the `None` element Pandas returns `None` — no exception, no cascade failure. The same is true for `s.str.upper()`, which uppercases everything present and keeps missing entries missing. This is the single most practical detail in the session: with vanilla Python, one stray `None` kills your loop; with `.str`, missing values ride through quietly and propagate as `None`/`NaN` instead of crashing.

The magical property is that the accessor gives you *every* standard string method in vectorised form — `upper`, `lower`, `capitalize`, `title`, `strip`, `split`, `replace`, `contains`, `startswith`, `endswith`, `isdigit`, and regex-capable ones — all usable on the literal strings
 as if they lived in one giant C loop. Being vectorised, they are fast; being Pandas-native, they compose with the rest of the ecosystem: filter using the resulting boolean mask, chain accessors (`s.str.lower().str.strip()`), and never write a Python loop for text again. After this warm-up, the notebook moves to a real column — Titanic passenger names — where the power becomes obvious.

In [21]:
# How pandas solves this issue?

s = pd.Series(['cat','mat',None,'rat'])
# string accessor
s.str.startswith('c')

# fast and optimized

0     True
1    False
2     None
3    False
dtype: object

In [23]:
# import titanic
df = pd.read_csv('/content/titanic.csv')
df['Name']

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

# 3. Common String Methods

The Titanic `Name` column follows `Lastname, Title. Firstnames`. It is perfect raw material for practising case conversion, splitting, and slicing.

## 📖 Common string methods

```python
df['Name'].str.upper()                     # case
df['Name'].str.lower()
df['Name'].str.capitalize()
df['Name'].str.title()
df['lastname'] = df['Name'].str.split(',').str.get(0)  # split
```

- `lower/upper/capitalize/title` — case uniform
- `split(',')` → list; `.str.get(i)` → element
- `.str.strip()` — remove whitespace
- Chain: `split → get → strip → split`


## 🔬 Deep Dive : Common string methods

```python
df['Name'].str.upper() / .lower() / .title() / .capitalize()
df['lastname'] = df['Name'].str.split(',').str.get(0)
df[['title','firstname']] = df['Name'].str.split(',') \
                                .str.get(1).str.strip() \
                                .str.split(' ', n=1, expand=True)
```

- case: upper/lower/title/capitalize — uniform text
- split(',') → Series of LISTS
- .str.get(i) → pick i-th element
- .str.strip() — remove leading/trailing spaces
- split with expand=True + n=1 → DataFrame multiple cols
- chain accessors — pipeline one-shot
- .str.len(), .str.contains('x'), .str.replace()
- titanic Name parsing: lastname/title/firstname extraction
- All vectorized, handle NaN
- Cleaning: normalize whitespace & case before matching

## 3.1 Upper, lower, capitalize, title, and split

Case normalisation is the bread-and-butter of cleaning. `df['Name'].str.upper()` makes every character uppercase, `.str.lower()` the reverse, `.str.capitalize()` capitalises only the first letter of each string, and `.str.title()` capitalises the first letter of every word. They matter because 'john', 'John', and 'JOHN' should match as the same person when you filter or group — but only after you standardise the case.

Length and whitespace are next. `df['Name'][df['Name'].str.len() == 82].values[0]` finds the single longest name (82 characters) and returns it — `.str.len()` counts characters element-wise. The bare Python `"   nitish   ".strip()` demonstrates how `strip` removes surrounding spaces; `df['Name'].str.strip()` applies it to every name, guarding against ragged whitespace left by CSVs.

The headline trick is splitting a Titanic name into parts. `df['Name'].str.split(',')` turns each name into a list, e.g. `Braund, Mr. Owen Harris` → `['Braund', ' Mr. Owen Harris']`. Follow with `.str.get(0)` to keep element 0 of each list — that is the surname, stored into `df['lastname']`. Then the second element is stripped and re-split on spaces with `n=1, expand=True`, fanning one string into two DataFrame columns `title` and `firstname` in a single assignment. `value_counts()` on the extracted titles immediately reveals the passenger mix — the power of chaining accessors.

In [35]:
# Common Functions
# lower/upper/capitalize/title
df['Name'].str.upper()
df['Name'].str.capitalize()
df['Name'].str.title()
# len
df['Name'][df['Name'].str.len() == 82].values[0]
# strip
"                   nitish                              ".strip()
df['Name'].str.strip()

0                                Braund, Mr. Owen Harris
1      Cumings, Mrs. John Bradley (Florence Briggs Th...
2                                 Heikkinen, Miss. Laina
3           Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                               Allen, Mr. William Henry
                             ...                        
886                                Montvila, Rev. Juozas
887                         Graham, Miss. Margaret Edith
888             Johnston, Miss. Catherine Helen "Carrie"
889                                Behr, Mr. Karl Howell
890                                  Dooley, Mr. Patrick
Name: Name, Length: 891, dtype: object

In [41]:
# split -> get
df['lastname'] = df['Name'].str.split(',').str.get(0)
df.head()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked   lastname  
0      0         A/5 21171   7.2500   NaN        S     Braund  
1      0          PC 17599  71.2833   C85        C    Cumings  
2      0  STON/O2. 3101282   7.9250   NaN        S  Heikkinen  
3      0            113803  53.1000  C123        S   Futrelle  


In [59]:
df[['title','firstname']] = df['Name'].str.split(',').str.get(1).str.strip().str.split(' ', n=1, expand=True)
df.head()

df['title'].value_counts()

Mr.          517
Miss.        182
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Mlle.          2
Major.         2
Col.           2
the            1
Capt.          1
Ms.            1
Sir.           1
Lady.          1
Mme.           1
Don.           1
Jonkheer.      1
Name: title, dtype: int64

# 4. Cleaning Titles and Filtering Text

Theory turns to action: standardise the messy `title` values, then use text conditions as row filters.

### Functional tools
map/filter apply functions; reduce aggregates. Often paired with lambdas.

## 📖 replace & filtering

```python
df['title'] = df['title'].str.replace('Ms.', 'Miss.')
df[df['firstname'].str.endswith('A')]       # filter rows
df[df['firstname'].str.startswith(...)]
df['Name'].str[::-1]                         # reverse slice
```

- `replace(old, new)` — substring replace
- `startswith/endswith` — boolean filter
- `contains(regex)` — pattern search (regex support)
- String slicing `[::-1]` also works


## 🔬 Deep Dive : replace & filtering strings

```python
df['title'] = df['title'].str.replace('Ms.','Miss.')   # exact string swap
df[df['firstname'].str.endswith('A')]          # filter rows
df[df['firstname'].str.startswith('William')]  # prefix rows
```

- replace(old, new) — string substitute (exact substring by default, regex=True option)
- startswith / endswith — prefix/suffix filter → bool mask → df[mask]
- combine: df[df['name'].str.contains('Mr.', na=False)]
- `na=False` keeps rows where NaN doesn't get dropped by comparison
- filter = the main string use in analysis (name, city, id patterns)
- pattern matching both edge-based and contains-based
- output keeps full df, only filters rows
- value_counts on cleaned titles — insight after cleanup
- boolean masks compose with & | ~

## 4.1 replace, startswith, endswith, isdigit

Two Titanic quirks get fixed first: `Ms.` and `Mlle.` are the same person-title as `Miss.`. `df['title'].str.replace('Ms.','Miss.')` swaps the substring everywhere, and repeating it for `Mlle.` merges the counts — check `value_counts()` afterwards and `Miss.` has grown from 182 to 185. By default `replace` treats the search text as a **regular expression**, which is why Pandas emits a `FutureWarning` promising the default will flip to plain-text in a future release; for literal swaps you can pass `regex=False`.

Filtering rows by their text is the second trick. `df[df['firstname'].str.endswith('A')]` keeps only passengers whose first name ends in 'A' — the boolean Series from `.endswith` becomes a mask that selects whole rows. `.str.isdigit()` similarly checks whether the name is made entirely of digits; on beginners' data it returns an empty frame here, but on dirty data it is the fastest way to find rows where numbers got typed into a text column. The general pattern — `df[df['col'].str.<condition>()]` — is one of the most repeated lines in Pandas analysis because it composes: prefix checks with `startswith`, suffix checks with `endswith`, and substring checks with `contains`. Every result keeps the full DataFrame intact and simply filters rows.

In [60]:
# replace
df['title'] = df['title'].str.replace('Ms.','Miss.')
df['title'] = df['title'].str.replace('Mlle.','Miss.')

<ipython-input-60-9403e5934d1f>:2: FutureWarning: The default value of regex will change from True to False in a future version.
  df['title'] = df['title'].str.replace('Ms.','Miss.')
<ipython-input-60-9403e5934d1f>:3: FutureWarning: The default value of regex will change from True to False in a future version.
  df['title'] = df['title'].str.replace('Mlle.','Miss.')


In [61]:
df['title'].value_counts()

Mr.          517
Miss.        185
Mrs.         125
Master.       40
Dr.            7
Rev.           6
Major.         2
Col.           2
Don.           1
Mme.           1
Lady.          1
Sir.           1
Capt.          1
the            1
Jonkheer.      1
Name: title, dtype: int64

In [66]:
# filtering
# startswith/endswith
df[df['firstname'].str.endswith('A')]
# isdigit/isalpha...
df[df['firstname'].str.isdigit()]

Empty DataFrame
Columns: [PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked, lastname, title, firstname]
Index: []

# 5. Regular Expressions with .str

When `startswith` and `contains` are not expressive enough, regular expressions step in: a tiny pattern language that describes whole classes of text.

### Key ideas
Classes/objects model state+behavior. Encapsulation hides internals; abstraction focuses on essentials. Inheritance/polymorphism extend reuse.

## 📖 Regex with .str

```python
df[df['firstname'].str.contains('John', case=False)]
```

- `contains` — regex pattern match
- `case=False` — case-insensitive
- Filtering = boolean mask
- Common in data cleaning (irregular text)


## 🔬 Deep Dive : Regex with .str

```python
# contains -> search substring or pattern
df[df['firstname'].str.contains('john', regex=True)]              # case sensitive
df[df['firstname'].str.contains('john|John', regex=True)]         # alternation
df[df['firstname'].str.contains('.*john.*', flags=re.I, regex=True)]  # ignorecase whole string
df['Name'].str[::-1]     # reverse each name (slicing on strings!)
```

- str.contains(pattern, regex=True) — substring or regex test
- flags=re.I — case-insensitive
- alternation |, wildcards ., anchors ^ $, [a-z] classes
- str[::-1] — python string slicing applied elementwise
- str.findall/extract for capture groups (complex)
- regex = pattern languages for text matching
- Filter by patterns — phone, email, code columns
- escape specials, test carefully
- regex slows down — simple substring first via regex=False
- Every text col query pattern starts here

## 5.1 Regex filtering in a single line

`.str.contains('john', case=False)` is the workhorse: it tests every first name for the substring 'john' ignoring case, so both 'John' and 'john' match, and returns a boolean mask you feed straight into `df[...]`. Because `contains` supports regex by default, the pattern can grow arbitrarily powerful: `'john|John'` matches either spelling via alternation, and anchors like `^` and `$` pin the pattern to the start/end of the string.

The lastname filter demonstrates real regex muscle: `df[df['lastname'].str.contains('^[^aeiouAEIOU].+[^aeiouAEIOU]$')]` keeps only surnames that begin *and* end with a consonant — `^[^aeiou...]` matches a leading non-vowel, `.+` any middle, and `[^aeiou...]$` a trailing non-vowel — parsing the pattern visually is the skill to practise.

Two closing notes. First, if you only need a literal substring, pass `regex=False`; plain-text search is dramatically faster than pattern matching on millions of rows. Second, remember `.str` is a general accessor: `df['Name'].str[::-1]` slices every string from right to left, returning each name reversed — standard Python string slicing applied element-wise. Between `contains`, `case=False`, anchored patterns, and slicing, every text column in a dataset becomes searchable and filterable in one expressive line.

In [72]:
# applying regex
# contains
# search john -> both case
df[df['firstname'].str.contains('john',case=False)]
# find lastnames with start and end char vowel
df[df['lastname'].str.contains('^[^aeiouAEIOU].+[^aeiouAEIOU]$')]

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
5              6         0       3   
6              7         0       1   
..           ...       ...     ...   
884          885         0       3   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
5                                     Moran, Mr. James    male   NaN      0   
6                              McCarthy, Mr. Timothy J    male  54.0      0   
..                                                 ...     ...   ... 

In [76]:
# slicing
df['Name'].str[::-1]

0                                sirraH newO .rM ,dnuarB
1      )reyahT sggirB ecnerolF( yeldarB nhoJ .srM ,sg...
2                                 aniaL .ssiM ,nenikkieH
3           )leeP yaM yliL( htaeH seuqcaJ .srM ,ellertuF
4                               yrneH mailliW .rM ,nellA
                             ...                        
886                                sazouJ .veR ,alivtnoM
887                         htidE teragraM .ssiM ,maharG
888             "eirraC" neleH enirehtaC .ssiM ,notsnhoJ
889                                llewoH lraK .rM ,rheB
890                                  kcirtaP .rM ,yelooD
Name: Name, Length: 891, dtype: object

## Summary

This session turned text wrangling from a slow, crash-prone loop into fast, vectorised expressions. You saw why NumPy-style vectorisation wins: one operation over the whole column instead of a Python loop, and you saw the classic failure — `None.startswith(...)` raises `TypeError` — that vanilla Python cannot survive. Pandas' **`.str` accessor** solved it: `s.str.startswith('c')` and `s.str.upper()` run across the column and merely propagate missing values. Using Titanic names you converted case with `upper/lower/capitalize/title`, measured length with `.str.len()`, stripped whitespace, exploded `Lastname, Title. Firstname` into separate columns via `split(...).str.get(...)` and `expand=True`, and counted the extracted titles. Cleaning continued with `str.replace` (merging Ms./Mlle. into Miss.), and filtering joined the toolkit through `startswith`, `endswith`, and `isdigit` boolean masks. Finally, **regular expressions** arrived: `contains('john', case=False)`, alternation with `|`, anchored consonant-consonant surname patterns, and element-wise string slicing with `[::-1]` — a complete vocabulary for searching and tidying any text column.